In [14]:
!pip install beautifulsoup4

In [15]:
import os
import json
import time
import random
import zipfile
import requests
import pandas as pd
from bs4 import BeautifulSoup

# Class Explanation: `NewsScraper`

## Overview
The `NewsScraper` class is designed for scraping news articles from three different Urdu news websites: Geo, Jang, and Express. The class has methods that cater to each site's unique structure and requirements. Below, we will go through the class and its methods, detailing what each function does, the input it takes, and the output it returns.

## Class Definition

```python
class NewsScraper:
    def __init__(self, id_=0):
        self.id = id_
```


## Method 1: `get_express_articles`

### Description
Scrapes news articles from the Express website across categories like saqafat (entertainment), business, sports, science-technology, and world. The method navigates through multiple pages for each category to gather a more extensive dataset.

### Input
- **`max_pages`**: The number of pages to scrape for each category (default is 7).

### Process
- Iterates over each category and page.
- Requests each category page and finds article cards within `<ul class='tedit-shortnews listing-page'>`.
- Extracts the article's headline, link, and content by navigating through `<div class='horiz-news3-caption'>` and `<span class='story-text'>`.

### Output
- **Returns**: A tuple of:
  - A Pandas DataFrame containing columns: `id`, `title`, and `link`).
  - A dictionary `express_contents` where the key is the article ID and the value is the article content.

### Data Structure
- Article cards are identified by `<li>` tags.
- Content is structured within `<span class='story-text'>` and `<p>` tags.



In [16]:
class NewsScraper:
    def __init__(self,id_=0):
        self.id = id_


  # write functions to scrape from other websites


    def get_express_articles(self, max_pages=7):
        express_df = {
            "id": [],
            "title": [],
            "link": [],
            "content": [],
            "gold_label": [],
        }
        base_url = 'https://www.express.pk'
        categories = ['saqafat', 'business', 'sports', 'science', 'world']   # saqafat is entertainment category

        # Iterating over the specified number of pages
        for category in categories:
            for page in range(1, max_pages + 1):
                print(f"Scraping page {page} of category '{category}'...")
                url = f"{base_url}/{category}/archives?page={page}"
                response = requests.get(url)
                response.raise_for_status()
                soup = BeautifulSoup(response.text, "html.parser")

                # Finding article cards
                cards = soup.find('ul', class_='tedit-shortnews listing-page').find_all('li')  # Adjust class as per actual site structure
                print(f"\t--> Found {len(cards)} articles on page {page} of '{category}'.")

                success_count = 0

                for card in cards:
                    try:
                        div = card.find('div',class_='horiz-news3-caption')

                        # Article Title
                        headline = div.find('a').get_text(strip=True).replace('\xa0', ' ')

                        # Article link
                        link = div.find('a')['href']

                        # Requesting the content from each article's link
                        article_response = requests.get(link)
                        article_response.raise_for_status()
                        content_soup = BeautifulSoup(article_response.text, "html.parser")


                        # Content arranged in paras inside <span> tags
                        paras = content_soup.find('span',class_='story-text').find_all('p')

                        combined_text = " ".join(
                        p.get_text(strip=True).replace('\xa0', ' ').replace('\u200b', '')
                        for p in paras if p.get_text(strip=True)
                        )

                        # Storing data
                        express_df['id'].append(self.id)
                        express_df['title'].append(headline)
                        express_df['link'].append(link)
                        express_df['gold_label'].append(category.replace('saqafat','entertainment').replace('science','science-technology'))
                        express_df['content'].append(combined_text)

                        # Increment ID and success count
                        self.id += 1
                        success_count += 1

                    except Exception as e:
                        print(f"\t--> Failed to scrape an article on page {page} of '{category}': {e}")

                print(f"\t--> Successfully scraped {success_count} articles from page {page} of '{category}'.")
            print('')

        return pd.DataFrame(express_df)
    

   
    def get_geo_articles(self):
        geo_df = {
            "id": [],
            "title": [],
            "link": [],
            "content": [],
            "gold_label": [],
        }
        base_url = 'https://urdu.geo.tv/category/'
        categories = ['entertainment', 'business', 'sports', 'science-technology', 'world']

        
        for category in categories:
            print(f"Scraping category '{category}'...")
            url = f"{base_url}{category}"
            response = requests.get(url)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            
            cards = soup.find_all('li', class_='border-box')
            print(f"\t--> Found {len(cards)} articles in '{category}'.")

            success_count = 0

            for card in cards:
                try:
                    
                    headline_tag = card.find('div', class_='entry-title')
                    headline = headline_tag.find('h2').get_text(strip=True)

                  
                    link = card.find('a', class_='open-section')['href']
                    full_link = f"https://urdu.geo.tv{link}" if not link.startswith('http') else link

                    
                    article_response = requests.get(full_link)
                    article_response.raise_for_status()
                    article_soup = BeautifulSoup(article_response.text, "html.parser")

                    
                    content_div = article_soup.find('div', class_='content-area')
                    if content_div:
                        paragraphs = content_div.find_all('p')
                        combined_text = " ".join(
                            p.get_text(strip=True).replace('\xa0', ' ').replace('\u200b', '')
                            for p in paragraphs if p.get_text(strip=True)
                        )
                    else:
                        combined_text = "Content not available."

                    
                    geo_df['id'].append(self.id)
                    geo_df['title'].append(headline)
                    geo_df['link'].append(full_link)
                    geo_df['gold_label'].append(category)
                    geo_df['content'].append(combined_text)

                    
                    self.id += 1
                    success_count += 1

                except Exception as e:
                    print(f"\t--> Failed to scrape an article from '{category}': {e}")

            print(f"\t--> Successfully scraped {success_count} articles from '{category}'.")
            print('')

        return pd.DataFrame(geo_df)
    
    def get_jang_articles(self):
        jang_df = {
            "id": [],
            "title": [],
            "link": [],
            "content": [],
            "gold_label": [],
        }
        base_url = 'https://jang.com.pk/category/latest-news/'
        categories = ['entertainment', 'business', 'sports', 'health-science', 'world']

        
        for category in categories:
            print(f"Scraping articles from category '{category}'...")
            url = f"{base_url}{category}"
            response = requests.get(url)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            # Finding article cards
            cards = soup.find_all('div', class_='main-heading')
            print(f"\t--> Found {len(cards)} articles in category '{category}'.")

            success_count = 0

            for card in cards:
                try:
                    
                    headline_tag = card.find('a')
                    headline = headline_tag.find('h2').get_text(strip=True).replace('\xa0', ' ')
                    link = headline_tag['href']

                   
                    article_response = requests.get(link)
                    article_response.raise_for_status()
                    content_soup = BeautifulSoup(article_response.text, "html.parser")

                    
                    paras = content_soup.find('div', class_='detail_view_content').find_all('p')
                    combined_text = " ".join(
                        p.get_text(strip=True).replace('\xa0', ' ').replace('\u200b', '')
                        for p in paras if p.get_text(strip=True)
                    )

                   
                    jang_df['id'].append(self.id)
                    jang_df['title'].append(headline)
                    jang_df['link'].append(link)
                    jang_df['gold_label'].append(category.replace('health-science', 'science-technology'))
                    jang_df['content'].append(combined_text)

                    
                    self.id += 1
                    success_count += 1

                except Exception as e:
                    print(f"\t--> Failed to scrape an article in category '{category}': {e}")

            print(f"\t--> Successfully scraped {success_count} articles from category '{category}'.")
            print('')

        return pd.DataFrame(jang_df)
    
    def get_samaa_articles(self, max_pages=3):
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/111.0.0.0 Safari/537.36"
        }
        samaa_df = {
            "id": [],
            "title": [],
            "link": [],
            "content": [],
            "gold_label": [],
        }
        base_url = 'https://urdu.samaa.tv'
        categories = ['lifestyle', 'money', 'sports', 'tech', 'global']

        for category in categories:
            print(f"Scraping category '{category}'...")
            success_count = 0

            for page in range(1, max_pages + 1):
                print(f"  Scraping page {page} of category '{category}'...")
                url = f"{base_url}/{category}?page={page}"
                try:
                    response = requests.get(url, headers=headers)
                    response.raise_for_status()
                except requests.exceptions.HTTPError as e:
                    print(f"Failed to scrape {url}: {e}")
                    break  

                soup = BeautifulSoup(response.text, "html.parser")

               
                articles = soup.find_all('div', class_='text')
                print(f"\t--> Found {len(articles)} articles on page {page} of '{category}'.")

                for article in articles:
                    try:
                       
                        title_tag = article.find('h3').find('a')
                        title = title_tag.get_text(strip=True)
                        link = title_tag['href']

                      
                        if not link.startswith('http'):
                            link = f"{base_url}{link}"

                       
                        article_response = requests.get(link, headers=headers)
                        article_response.raise_for_status()
                        content_soup = BeautifulSoup(article_response.text, "html.parser")

                      
                        article_div = content_soup.find('div', class_='article-content')
                        if article_div:
                            paras = article_div.find_all('p')
                            content = " ".join(
                                (p.get_text(strip=True) or "").replace('\xa0', ' ').replace('\u200b', '')
                                for p in paras if p.get_text(strip=True))
                        else:
                            content = "" 

                        samaa_df['id'].append(self.id)
                        samaa_df['title'].append(title)
                        samaa_df['link'].append(link)
                        samaa_df['gold_label'].append(category)
                        samaa_df['content'].append(content)
                        
                        self.id += 1
                        success_count += 1
                        print("scrapped",success_count)

                    except Exception as e:
                        print(f"\t--> Failed to scrape an article on page {page} of '{category}': {e}")

                # Break if no articles are found on the current page
                if not articles:
                    print(f"\t--> No more articles found on page {page} of '{category}'. Stopping pagination.")
                    break

            print(f"\t--> Successfully scraped {success_count} articles from category '{category}'.")
            print('')

        return pd.DataFrame(samaa_df)
    
    def get_dawn_articles(self):
        dawn_df = {
            "id": [],
            "title": [],
            "link": [],
            "content": [],
            "gold_label": [],
        }
        base_url = 'https://www.dawnnews.tv/'
        categories = ['life-style', 'business', 'sport', 'tech', 'world']  

        for category in categories:
            print(f"Scraping category '{category}'...")
            url = f"{base_url}{category}"
            response = requests.get(url)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            
            cards = soup.find_all('div', class_='col-span-7')
            print("no of cards",len(cards))
            print(f"\t--> Found {len(cards)} articles in category '{category}'.")

            success_count = 0

            for card in cards:
                try:
                
                    title = card.find('h2', class_='story__title').get_text(strip=True)
                    link = card.find('h2', class_='story__title').find('a')['href']

                   
                    article_response = requests.get(link)
                    article_response.raise_for_status()
                    content_soup = BeautifulSoup(article_response.text, "html.parser")

                  
                    paras = content_soup.find('div', class_='story__content').find_all('p')
                    combined_text = " ".join(
                        p.get_text(strip=True).replace('\xa0', ' ').replace('\u200b', '')
                        for p in paras if p.get_text(strip=True)
                    )

                  
                    dawn_df['id'].append(self.id)
                    dawn_df['title'].append(title)
                    dawn_df['link'].append(link)
                    dawn_df['gold_label'].append(category)
                    dawn_df['content'].append(combined_text)

                    
                    self.id += 1
                    success_count += 1
                    print("scrapped",success_count)

                except Exception as e:
                    print(f"\t--> Failed to scrape an article in category '{category}': {e}")

            print(f"\t--> Successfully scraped {success_count} articles from category '{category}'.")
            print('')

        return pd.DataFrame(dawn_df)
    
    def get_dunya_articles(self):
        dunya_df = {
            "id": [],
            "title": [],
            "link": [],
            "content": [],
            "gold_label": [],
        }
        base_url = 'https://urdu.dunyanews.tv/index.php'
        categories = ['/ur/Entertainment', '/ur/Business', '/ur/Sports', '/en/Technology', '/ur/World']

        for category in categories:
            print(f"Scraping category '{category}'...")
            url = f"{base_url}{category}"
            response = requests.get(url)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            # Finding article cards
            cards = soup.find_all('div', class_='cNewsBox')
            print(f"\t--> Found {len(cards)} articles in category '{category}'.")

            success_count = 0

            for card in cards:
                try:
                    
                    title = card.find('h3').get_text(strip=True)
                    link = f"https://urdu.dunyanews.tv{card.find('h3').find('a')['href']}"

                    article_response = requests.get(link)
                    article_response.raise_for_status()
                    content_soup = BeautifulSoup(article_response.text, "html.parser")

                  
                    paras = content_soup.find('div', class_='main-news').find_all('p')
                    combined_text = " ".join(
                        p.get_text(strip=True).replace('\xa0', ' ').replace('\u200b', '')
                        for p in paras if p.get_text(strip=True)
                    )

         
                    dunya_df['id'].append(self.id)
                    dunya_df['title'].append(title)
                    dunya_df['link'].append(link)
                    dunya_df['gold_label'].append(category.replace('/ur/Entertainment', 'entertainment').replace('/en/Technology', 'science-technology').replace('/ur/World', 'world').replace('/ur/Business', 'business').replace('/ur/Sports', 'sports'))
                    dunya_df['content'].append(combined_text)

                   
                    self.id += 1
                    success_count += 1

                except Exception as e:
                    print(f"\t--> Failed to scrape an article in category '{category}': {e}")

            print(f"\t--> Successfully scraped {success_count} articles from category '{category}'.")
            print('')

        return pd.DataFrame(dunya_df)



       


    






        
                
            

In [11]:
scraper = NewsScraper()

In [5]:
express_df = scraper.get_express_articles()

Scraping page 1 of category 'saqafat'...
	--> Found 10 articles on page 1 of 'saqafat'.
	--> Successfully scraped 10 articles from page 1 of 'saqafat'.
Scraping page 2 of category 'saqafat'...
	--> Found 10 articles on page 2 of 'saqafat'.
	--> Successfully scraped 10 articles from page 2 of 'saqafat'.
Scraping page 3 of category 'saqafat'...
	--> Found 10 articles on page 3 of 'saqafat'.
	--> Successfully scraped 10 articles from page 3 of 'saqafat'.
Scraping page 4 of category 'saqafat'...
	--> Found 10 articles on page 4 of 'saqafat'.
	--> Successfully scraped 10 articles from page 4 of 'saqafat'.
Scraping page 5 of category 'saqafat'...
	--> Found 10 articles on page 5 of 'saqafat'.
	--> Successfully scraped 10 articles from page 5 of 'saqafat'.
Scraping page 6 of category 'saqafat'...
	--> Found 10 articles on page 6 of 'saqafat'.
	--> Successfully scraped 10 articles from page 6 of 'saqafat'.
Scraping page 7 of category 'saqafat'...
	--> Found 10 articles on page 7 of 'saqafat'.


In [9]:
print(express_df.head())    

   id                                              title  \
0   0  رنبیر کپور کا انکشاف: عالیہ نے پوچھا، 'کشور کم...   
1   1  فلم ’مہاراج‘ پر قانونی مسائل سے پریشان ہوگئے ت...   
2   2  "دی نائٹ مینیجر" انٹرنیشنل ایمی ایوارڈز کیلئے ...   
3   3  حنا خان کا سلمان خان کے لیے جذباتی نوٹ: "آپ نے...   
4   4  کیا ورن دھون نے صرف 4 دن بعد اپنا لنکڈ اِن اکا...   

                                                link  \
0  https://www.express.pk/story/2734883/ranbirkap...   
1  https://www.express.pk/story/2734882/film-maha...   
2  https://www.express.pk/story/2734875/thenightm...   
3  https://www.express.pk/story/2734871/hinakhan-...   
4  https://www.express.pk/story/2734867/kia-varun...   

                                             content     gold_label  
0  بالی ووڈ اداکار رنبیر کپور نے انٹرنیشنل فلم فی...  entertainment  
1  بالی ووڈ کے سپر اسٹار عامر خان کے بیٹے جنید خا...  entertainment  
2  مشہور اداکار ادیتیہ رائے کپور کی پہلی او ٹی ٹی...  entertainment  
3  بالی ووڈ اداکارہ حن

In [6]:
geo_df = scraper.get_geo_articles()

Scraping category 'entertainment'...
	--> Found 60 articles in 'entertainment'.
	--> Successfully scraped 60 articles from 'entertainment'.

Scraping category 'business'...
	--> Found 60 articles in 'business'.
	--> Successfully scraped 60 articles from 'business'.

Scraping category 'sports'...
	--> Found 60 articles in 'sports'.
	--> Successfully scraped 60 articles from 'sports'.

Scraping category 'science-technology'...
	--> Found 60 articles in 'science-technology'.
	--> Successfully scraped 60 articles from 'science-technology'.

Scraping category 'world'...
	--> Found 60 articles in 'world'.
	--> Successfully scraped 60 articles from 'world'.



In [7]:
print(geo_df.head())

    id                                              title  \
0  350  ' یہ بھی سی وی بھیجیں گے'، صارفین کی تنقید کے ...   
1  351  ویڈیو: اسلام آباد کے جوڑے نے بند راستوں اور ک...   
2  352  میرے نزدیک دوستی ضروری، شادی بیکار کام ہے: جاو...   
3  353  اہلیہ ایشوریا کیلئے ابھیشیک کے بیان نے مداحوں ...   
4  354  کراچی: سندھ کے قدیم فنون کو اجاگر کرتا تین روز...   

                                 link  \
0  https://urdu.geo.tv/latest/388031-   
1  https://urdu.geo.tv/latest/388006-   
2  https://urdu.geo.tv/latest/387983-   
3  https://urdu.geo.tv/latest/387973-   
4  https://urdu.geo.tv/latest/387953-   

                                             content     gold_label  
0  بالی وڈ اداکار ورون دھون نے پروفیشنل نیٹ ورکنگ...  entertainment  
1  اسلام آباد سمیت راولپنڈی اور لاہور میں پاکستا...  entertainment  
2  بھارتی مصنف اور نغمہ نگار   جاوید اختر   نے اد...  entertainment  
3  بالی وڈ اسٹار کپل کے درمیان  گزشتہ کئی عرصے سے...  entertainment  
4  سندھ کی قدیم تہذیب، تاریخ، روایات

In [8]:
jang_df = scraper.get_jang_articles()  

Scraping articles from category 'entertainment'...
	--> Found 104 articles in category 'entertainment'.
	--> Failed to scrape an article in category 'entertainment': 'NoneType' object has no attribute 'find'
	--> Failed to scrape an article in category 'entertainment': 'NoneType' object has no attribute 'find'
	--> Failed to scrape an article in category 'entertainment': 'NoneType' object has no attribute 'find'
	--> Failed to scrape an article in category 'entertainment': 'NoneType' object has no attribute 'find'
	--> Successfully scraped 100 articles from category 'entertainment'.

Scraping articles from category 'business'...
	--> Found 101 articles in category 'business'.
	--> Failed to scrape an article in category 'business': 'NoneType' object has no attribute 'find'
	--> Failed to scrape an article in category 'business': 'NoneType' object has no attribute 'find'
	--> Failed to scrape an article in category 'business': 'NoneType' object has no attribute 'find'
	--> Failed to scr

In [10]:
print(jang_df["id"][0])

650


In [17]:
samaa_df = scraper.get_samaa_articles() 


Scraping category 'lifestyle'...
  Scraping page 1 of category 'lifestyle'...
	--> Found 25 articles on page 1 of 'lifestyle'.
scrapped 1
scrapped 2
scrapped 3
scrapped 4
scrapped 5
scrapped 6
scrapped 7
scrapped 8
scrapped 9
scrapped 10
scrapped 11
scrapped 12
scrapped 13
scrapped 14
scrapped 15
scrapped 16
scrapped 17
scrapped 18
scrapped 19
scrapped 20
scrapped 21
scrapped 22
scrapped 23
scrapped 24
scrapped 25
  Scraping page 2 of category 'lifestyle'...
	--> Found 25 articles on page 2 of 'lifestyle'.
scrapped 26
scrapped 27
scrapped 28
scrapped 29
scrapped 30
scrapped 31
scrapped 32
scrapped 33
scrapped 34
scrapped 35
scrapped 36
scrapped 37
scrapped 38
scrapped 39
scrapped 40
scrapped 41
scrapped 42
scrapped 43
scrapped 44
scrapped 45
scrapped 46
scrapped 47
scrapped 48
scrapped 49
scrapped 50
  Scraping page 3 of category 'lifestyle'...
	--> Found 25 articles on page 3 of 'lifestyle'.
scrapped 51
scrapped 52
scrapped 53
scrapped 54
scrapped 55
scrapped 56
scrapped 57
scrapped 5

In [18]:
samaa_df['gold_label'] = samaa_df['gold_label'].replace({'lifestyle': 'entertainment', 'money': 'business', 'tech': 'science-technology', 'global': 'world'})
print(samaa_df.head())
# in the gold kabel column, the categories are not consistent.
# I will replace the categories with the correct ones


    id                                              title  \
0  287  اداکار فہد مصطفیٰ کو برطانوی پارلیمنٹ میں2ایوا...   
1  288  بھارتی اداکارہ نرگس فخری کی بہن 2 افراد کو قتل...   
2  289  معروف بھارتی اداکارہ شوبیتھا شوانا نے خودکشی کرلی   
3  290  بھارتی اداکار وکرانت میسی نے اداکاری کو الوداع...   
4  291     سینئراداکارہ سویرا ندیم کی والدہ انتقال کرگئیں   

                               link  \
0  https://urdu.samaa.tv/2087325796   
1  https://urdu.samaa.tv/2087325726   
2  https://urdu.samaa.tv/2087325630   
3  https://urdu.samaa.tv/2087325618   
4  https://urdu.samaa.tv/2087325478   

                                             content     gold_label  
0  مقبول اداکار و ٹی وی میزبان فہد مصطفیٰ کو حال ...  entertainment  
1  بالی ووڈ کی مشہور زمانہ فلم ’راک اسٹار‘ سے شہر...  entertainment  
2  ساؤتھ انڈین فلم انڈسٹری سے تعلق رکھنے والی بھا...  entertainment  
3  بھارتی اداکار وکرانت میسی نے اداکاری سے ریٹائر...  entertainment  
4  پاکستان شوبز انڈسٹری کی سینئر اداکارہ سویرا ن

In [19]:
dawn_df = scraper.get_dawn_articles()

Scraping category 'life-style'...
no of cards 33
	--> Found 33 articles in category 'life-style'.
scrapped 1
scrapped 2
scrapped 3
scrapped 4
scrapped 5
scrapped 6
scrapped 7
scrapped 8
scrapped 9
scrapped 10
scrapped 11
scrapped 12
scrapped 13
scrapped 14
scrapped 15
scrapped 16
scrapped 17
scrapped 18
scrapped 19
scrapped 20
scrapped 21
scrapped 22
scrapped 23
	--> Failed to scrape an article in category 'life-style': 403 Client Error: Forbidden for url: https://www.dawnnews.tv/news/1247734/
	--> Failed to scrape an article in category 'life-style': 403 Client Error: Forbidden for url: https://www.dawnnews.tv/news/1247654/
	--> Failed to scrape an article in category 'life-style': 403 Client Error: Forbidden for url: https://www.dawnnews.tv/news/1247657/
	--> Failed to scrape an article in category 'life-style': 403 Client Error: Forbidden for url: https://www.dawnnews.tv/news/1247647/
	--> Failed to scrape an article in category 'life-style': 403 Client Error: Forbidden for url: htt

In [20]:
# the gold labels are not consistent with the other dataframes
# I will replace the categories with the correct ones
dawn_df['gold_label'] = dawn_df['gold_label'].replace({'life-style': 'entertainment', 'sport': 'sports', 'tech': 'science-technology'})
print(dawn_df.head())

    id                                              title  \
0  661  ایک اداکار نے شادی کی منتیں کرکے تھک جانے کے ب...   
1  662  چاہتی ہوں کوئی شاہ رخ خان سے متعلق سوال نہ کرے...   
2  663  توقیر ناصر کی والدہ، راحت کاظمی کی دادی بن چکی...   
3  664  سہیلیوں اور شوبز سے وابستہ طاقتور خواتین نے بھ...   
4  665  ہیروئن کاسٹ کرنے ’ہیرا منڈی‘ گئے، جہاں ریما کو...   

                                    link  \
0  https://www.dawnnews.tv/news/1248026/   
1  https://www.dawnnews.tv/news/1248013/   
2  https://www.dawnnews.tv/news/1248015/   
3  https://www.dawnnews.tv/news/1247977/   
4  https://www.dawnnews.tv/news/1247958/   

                                             content     gold_label  
0  گلوکارہ شازیہ منظور نے انکشاف کیا ہے کہ ایک اد...  entertainment  
1  ’سپر اسٹار‘ ماہرہ خان نے کہا ہے کہ وہ چاہتی ہی...  entertainment  
2  حال ہی میں اختتام پذیر ہونے والے مقبول ڈرامے ’...  entertainment  
3  اداکارہ ماورہ حسین نے انکشاف کیا ہے کہ حال ہی ...  entertainment  
4  ماضی کے مقبول ف

In [61]:
new_content_df=pd.concat([samaa_df,dawn_df],ignore_index=True)
new_content_df.to_csv('new_content.csv',index=False)

In [21]:
dunya_df = scraper.get_dunya_articles()

Scraping category '/ur/Entertainment'...
	--> Found 18 articles in category '/ur/Entertainment'.
	--> Successfully scraped 18 articles from category '/ur/Entertainment'.

Scraping category '/ur/Business'...
	--> Found 18 articles in category '/ur/Business'.
	--> Successfully scraped 18 articles from category '/ur/Business'.

Scraping category '/ur/Sports'...
	--> Found 18 articles in category '/ur/Sports'.
	--> Successfully scraped 18 articles from category '/ur/Sports'.

Scraping category '/en/Technology'...
	--> Found 18 articles in category '/en/Technology'.
	--> Successfully scraped 18 articles from category '/en/Technology'.

Scraping category '/ur/World'...
	--> Found 18 articles in category '/ur/World'.
	--> Successfully scraped 18 articles from category '/ur/World'.



In [22]:
print(dunya_df.head())


    id                                              title  \
0  725  عزیز میاں قوال کو دنیا سے رخصت ہوئے 24 برس بیت...   
1  726  نامور گلوکارہ روشن آراء بیگم کو مداحوں سے بچھ...   
2  727  نامور مزاح نگار اور شاعر پطرس بخاری کو بچھڑے 6...   
3  728  اداکارہ نرگِس نے فنکارہ نگار چودھری سمیت10 یوٹ...   
4  729  رجب بٹ پاکستان میں رواں سال سب سے زیادہ ٹاپ ٹر...   

                                                link  \
0  https://urdu.dunyanews.tv/index.php/ur/Enterta...   
1  https://urdu.dunyanews.tv/index.php/ur/Enterta...   
2  https://urdu.dunyanews.tv/index.php/ur/Enterta...   
3  https://urdu.dunyanews.tv/index.php/ur/Enterta...   
4  https://urdu.dunyanews.tv/index.php/ur/Enterta...   

                                             content     gold_label  
0  لاہور: (دنیا نیوز) رومانوی اور صوفیانہ کلام کو...  entertainment  
1  لاہور: (دنیا نیوز) نامور گلوکارہ روشن آراء بی...  entertainment  
2  لاہور: (دنیا نیوز) نامور مزاح نگار، افسانہ نگا...  entertainment  
3  لاہور:(ویب ڈی

In [68]:
dunya_df.to_csv('dunya.csv',index=False)

# Output
- Save a combined csv of all 3 sites.

In [11]:

final_df = pd.concat([express_df, geo_df, jang_df], ignore_index=True)
final_df.to_csv('second_batch_1.csv', index=False)


In [23]:
second_batch_2 = pd.concat([samaa_df, dawn_df, dunya_df], ignore_index=True)
second_batch_2.to_csv('second_batch_2.csv', index=False)